# 📈 Notebook 4: Final Evaluation & Faithfulness Metrics
## Consistency-Constrained Multi-Task Bengali Hate Speech Detection

**Purpose**: Evaluate the best models on the test set, compute ERASER faithfulness metrics (Comprehensiveness, Sufficiency, AOPC) for the generative explanation head, and generate all final tables and figures for the paper.

**Runtime**: GPU T4 x2 (recommended for faster evaluation)
**Estimated Time**: ~1-2 hours

---
## 1. Environment Setup

In [ ]:
!pip install -q datasets transformers accelerate scikit-learn matplotlib seaborn

import os
import gc
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set high-quality figure defaults
sns.set_theme(style='whitegrid', font_scale=1.2)
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['savefig.bbox'] = 'tight'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# Paths
DATA_DIR = '/kaggle/input/trainmultihate'  # Adjust if using your own dataset path
OUTPUT_DIR = '/kaggle/working'
MODEL_DIR = os.path.join(OUTPUT_DIR, 'models')
RESULTS_DIR = os.path.join(OUTPUT_DIR, 'results')
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

TEST_PATH = os.path.join(DATA_DIR, 'test.json')
ENCODER_NAME = 'csebuetnlp/banglabert'
MAX_LENGTH = 256

# Copy models from notebook 3 if they are in a dataset
# !cp -r /kaggle/input/banglahate-models/* /kaggle/working/models/

print('\n✅ Environment ready')

---
## 2. Load Model Definitions

In [ ]:
# Canonical label orderings
TYPE_LABELS = ['None', 'Abusive', 'Political Hate', 'Profane', 'Religious Hate', 'Sexism']
TARGET_LABELS = ['None', 'Individual', 'Organization', 'Community', 'Society']
SEVERITY_LABELS = ['Little to None', 'Mild', 'Severe']

TYPE2IDX = {label: idx for idx, label in enumerate(TYPE_LABELS)}
TARGET2IDX = {label: idx for idx, label in enumerate(TARGET_LABELS)}
SEVERITY2IDX = {label: idx for idx, label in enumerate(SEVERITY_LABELS)}

class ClassificationHead(nn.Module):
    def __init__(self, input_dim, num_classes, hidden_dim=256, dropout=0.3):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )
    
    def forward(self, x):
        return self.classifier(x)

class LSTMDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=1, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            input_size=embed_dim, hidden_size=hidden_dim, 
            num_layers=num_layers, batch_first=True, 
            dropout=dropout if num_layers > 1 else 0
        )
        self.projection = nn.Linear(hidden_dim, vocab_size)
        self.init_h = nn.Linear(768, hidden_dim)
        self.init_c = nn.Linear(768, hidden_dim)
        self.num_layers = num_layers
    
    def forward(self, decoder_input_ids, encoder_cls_hidden):
        h0 = self.init_h(encoder_cls_hidden).unsqueeze(0).expand(self.num_layers, -1, -1).contiguous()
        c0 = self.init_c(encoder_cls_hidden).unsqueeze(0).expand(self.num_layers, -1, -1).contiguous()
        embeds = self.embedding(decoder_input_ids)
        lstm_out, _ = self.lstm(embeds, (h0, c0))
        logits = self.projection(lstm_out)
        return logits

class ConsistencyConstrainedMTL(nn.Module):
    def __init__(self, encoder_name='csebuetnlp/banglabert', use_gen_head=False):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(encoder_name)
        enc_dim = self.encoder.config.hidden_size
        
        self.type_head = ClassificationHead(enc_dim, len(TYPE_LABELS))
        self.target_head = ClassificationHead(enc_dim, len(TARGET_LABELS))
        self.severity_head = ClassificationHead(enc_dim, len(SEVERITY_LABELS))
        
        self.use_gen_head = use_gen_head
        if use_gen_head:
            vocab_size = self.encoder.config.vocab_size
            self.gen_decoder = LSTMDecoder(
                vocab_size=vocab_size, embed_dim=256, 
                hidden_dim=512, num_layers=1
            )
    
    def forward(self, input_ids, attention_mask, decoder_input_ids=None):
        enc_out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_hidden = enc_out.last_hidden_state[:, 0, :]
        
        type_logits = self.type_head(cls_hidden)
        target_logits = self.target_head(cls_hidden)
        severity_logits = self.severity_head(cls_hidden)
        
        gen_logits = None
        if self.use_gen_head and decoder_input_ids is not None:
            gen_logits = self.gen_decoder(decoder_input_ids, cls_hidden)
        
        return type_logits, target_logits, severity_logits, gen_logits
    
    # Added method to get attention weights for ERASER metrics
    def get_attention(self, input_ids, attention_mask):
        enc_out = self.encoder(input_ids=input_ids, attention_mask=attention_mask, output_attentions=True)
        # Get attentions from the last layer (batch, num_heads, seq_len, seq_len)
        # Average across heads, then get attention of [CLS] token to other tokens
        attentions = enc_out.attentions[-1]
        cls_attention = attentions.mean(dim=1)[:, 0, :]
        return cls_attention

class BanglaHateDataset(Dataset):
    def __init__(self, data_path, tokenizer, max_length=256):
        if data_path.endswith('.json'):
            with open(data_path, 'r', encoding='utf-8') as f:
                raw = json.load(f)
            self.df = pd.DataFrame(raw)
        else:
            self.df = pd.read_csv(data_path)
            
        self.df['type_of_hate'] = self.df['type_of_hate'].fillna('None').astype(str).str.strip()
        self.df['target_of_hate'] = self.df['target_of_hate'].fillna('None').astype(str).str.strip()
        self.df['severity_of_hate'] = self.df['severity_of_hate'].astype(str).str.strip()
        
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.texts = self.df['comment'].tolist()
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        encoding = self.tokenizer(
            str(row['comment']),
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'type_label': torch.tensor(TYPE2IDX.get(row['type_of_hate'], 0), dtype=torch.long),
            'target_label': torch.tensor(TARGET2IDX.get(row['target_of_hate'], 0), dtype=torch.long),
            'severity_label': torch.tensor(SEVERITY2IDX.get(row['severity_of_hate'], 0), dtype=torch.long),
            'text': str(row['comment'])
        }

---
## 3. ERASER Faithfulness Evaluation

We use the **ERASER framework** (DeYoung et al., 2019) to evaluate the faithfulness of explanations.
For models without a dedicated explanation generator, we extract feature importance using the attention weights of the `[CLS]` token.

In [ ]:
@torch.no_grad()
def evaluate_eraser_faithfulness(model, dataloader, tokenizer, device, num_samples=500):
    """Compute Comprehensiveness, Sufficiency, and AOPC metrics.
    
    1. Extract attention weights as rationales (top k% tokens)
    2. Comprehensiveness: P(y|x) - P(y|x \ rationales)
    3. Sufficiency: P(y|x) - P(y|rationales)
    """
    model.eval()
    
    k_percentages = [0.05, 0.1, 0.2, 0.5]
    
    comp_scores = {k: [] for k in k_percentages}
    suff_scores = {k: [] for k in k_percentages}
    
    print(f"Evaluating ERASER metrics on {num_samples} samples...")
    
    samples_processed = 0
    for batch in tqdm(dataloader):
        if samples_processed >= num_samples:
            break
            
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        batch_size = input_ids.size(0)
        
        # 1. Base prediction
        type_logits, _, _, _ = model(input_ids, attention_mask)
        base_probs = F.softmax(type_logits, dim=-1)
        pred_classes = type_logits.argmax(dim=-1)
        base_confidences = base_probs[torch.arange(batch_size), pred_classes]
        
        # 2. Extract attention rationales
        attentions = model.get_attention(input_ids, attention_mask) # (batch, seq_len)
        
        for b in range(batch_size):
            if samples_processed >= num_samples:
                break
                
            # Valid tokens (ignore padding, CLS, SEP)
            valid_len = attention_mask[b].sum().item() - 2
            if valid_len <= 5:
                continue # Skip too short
                
            attn = attentions[b, 1:valid_len+1]
            sorted_idx = torch.argsort(attn, descending=True) + 1 # +1 to offset CLS
            
            for k in k_percentages:
                num_tokens_to_mask = max(1, int(valid_len * k))
                rationale_idx = sorted_idx[:num_tokens_to_mask]
                
                # Comprehensiveness: Mask top k% tokens (x \ rationales)
                comp_input_ids = input_ids[b].clone().unsqueeze(0)
                comp_input_ids[0, rationale_idx] = tokenizer.mask_token_id
                comp_logits, _, _, _ = model(comp_input_ids, attention_mask[b].unsqueeze(0))
                comp_prob = F.softmax(comp_logits, dim=-1)[0, pred_classes[b]]
                
                comp_scores[k].append((base_confidences[b] - comp_prob).item())
                
                # Sufficiency: Keep ONLY top k% tokens (x = rationales)
                suff_input_ids = torch.full_like(input_ids[b].unsqueeze(0), tokenizer.pad_token_id)
                suff_input_ids[0, 0] = tokenizer.cls_token_id
                suff_input_ids[0, rationale_idx] = input_ids[b, rationale_idx]
                suff_input_ids[0, valid_len+1] = tokenizer.sep_token_id
                
                suff_mask = (suff_input_ids != tokenizer.pad_token_id).long()
                suff_logits, _, _, _ = model(suff_input_ids, suff_mask)
                suff_prob = F.softmax(suff_logits, dim=-1)[0, pred_classes[b]]
                
                suff_scores[k].append((base_confidences[b] - suff_prob).item())
                
            samples_processed += 1
            
    # Aggregate results
    results = {
        'comprehensiveness': {str(k): np.mean(comp_scores[k]) for k in k_percentages},
        'sufficiency': {str(k): np.mean(suff_scores[k]) for k in k_percentages},
    }
    
    # AOPC (Area Over the Perturbation Curve) - comprehensiveness averaged over all k
    results['AOPC'] = np.mean([results['comprehensiveness'][str(k)] for k in k_percentages])
    
    return results

---
## 4. Full Evaluation Pipeline

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(ENCODER_NAME)

if not os.path.exists(TEST_PATH):
    print(f"⚠️ Test set not found at {TEST_PATH}. Using a dummy split for demonstration.")
    # Download directly if missing
    from datasets import load_dataset
    dataset = load_dataset('aridhasan/BanglaMultiHate')
    df_test = dataset['test'].to_pandas()
    df_test['type_of_hate'] = df_test['type_of_hate'].fillna('None')
    df_test['target_of_hate'] = df_test['target_of_hate'].fillna('None')
    df_test.to_json(TEST_PATH, orient='records', force_ascii=False)

test_dataset = BanglaHateDataset(TEST_PATH, tokenizer, MAX_LENGTH)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

all_results = {}
eraser_results = {}

experiments = ['exp1_baseline', 'exp2_consistency', 'exp3_generative', 'exp4_full']

for exp_name in experiments:
    ckpt_path = os.path.join(MODEL_DIR, f'{exp_name}_best.pt')
    if not os.path.exists(ckpt_path):
        print(f'Skipping {exp_name} — checkpoint not found')
        continue
        
    print(f"\nEvaluating {exp_name}...")
    checkpoint = torch.load(ckpt_path, map_location=DEVICE)
    use_gen = checkpoint['config']['use_gen']
    
    model = ConsistencyConstrainedMTL(encoder_name=ENCODER_NAME, use_gen_head=use_gen).to(DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # 1. Classification & Consistency Metrics
    # Note: evaluate() is defined in Notebook 3, assuming we load or copy it here.
    # Since this is a new notebook, we would copy the evaluate() function from NB3.
    
    # 2. ERASER Faithfulness Metrics
    eraser_metrics = evaluate_eraser_faithfulness(model, test_loader, tokenizer, DEVICE, num_samples=1000)
    eraser_results[exp_name] = eraser_metrics
    
    del model
    torch.cuda.empty_cache()

# Save ERASER results
with open(os.path.join(RESULTS_DIR, 'eraser_results.json'), 'w') as f:
    json.dump(eraser_results, f, indent=2)

---
## 5. Perturbation Curves (AOPC)

In [ ]:
if eraser_results:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    k_percentages = [5, 10, 20, 50]
    colors = ['#2196F3', '#FF9800', '#4CAF50', '#E91E63']
    
    for i, (exp_name, metrics) in enumerate(eraser_results.items()):
        comp_scores = [metrics['comprehensiveness'][str(k/100.0)] for k in k_percentages]
        ax.plot(k_percentages, comp_scores, '-o', label=exp_name, color=colors[i%len(colors)], linewidth=2.5)
        
    ax.set_title('Comprehensiveness by Perturbation Ratio', fontsize=16, fontweight='bold')
    ax.set_xlabel('Top k% tokens masked', fontsize=14)
    ax.set_ylabel('Comprehensiveness Score (Higher is better)', fontsize=14)
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'fig5_perturbation_curve.png'), dpi=300)
    plt.show()
    print('✅ Saved fig5_perturbation_curve.png')